In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/processed/orders_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
order_reviews = pd.read_csv("../data/processed/order_reviews_clean.csv")
customers = pd.read_csv("../data/processed/customers_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
order_payments = pd.read_csv("../data/processed/order_payments_clean.csv")
sellers = pd.read_csv("../data/processed/sellers_clean.csv")
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")
product_category_name_translations = pd.read_csv("../data/processed/category_translation_clean.csv")
closed_deals = pd.read_csv("../data/processed/closed_deals_clean.csv")
marketing_leads = pd.read_csv("../data/processed/marketing_leads_clean.csv")

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

I'm creating columns to allow me to perform analyses based on specific periods, such as years or months.

In [4]:
orders = orders.assign(
    purchase_year=orders["order_purchase_timestamp"].dt.year,
    purchase_month=orders["order_purchase_timestamp"].dt.month,
    purchase_day=orders["order_purchase_timestamp"].dt.day,
    purchase_hour=orders["order_purchase_timestamp"].dt.hour,
    purchase_weekday=orders["order_purchase_timestamp"].dt.day_name()
)

These features enable temporal analyses such as monthly sales trends, hourly purchasing behavior, and weekday purchasing patterns.

In [5]:
orders[
    [
        "purchase_month",
        "purchase_day",
        "purchase_hour",
        "purchase_weekday",
        "purchase_year"
    ]
].head()

,purchase_month,purchase_day,purchase_hour,purchase_weekday,purchase_year
0,10,2,10,Monday,2017
1,7,24,20,Tuesday,2018
2,8,8,8,Wednesday,2018
3,11,18,19,Saturday,2017
4,2,13,21,Tuesday,2018


I implement validation practices like this. Instead of just saying "the code ran," I examine whether the generated values ​​make sense.

In [6]:
orders["purchase_hour"].describe()

count    99441.000000
mean        14.770829
std          5.326800
min          0.000000
25%         11.000000
50%         15.000000
75%         19.000000
max         23.000000
Name: purchase_hour, dtype: float64

In [7]:
orders["purchase_month"].value_counts().sort_index()

purchase_month
1      8069
2      8508
3      9893
4      9343
5     10573
6      9412
7     10318
8     10843
9      4305
10     4959
11     7544
12     5674
Name: count, dtype: int64

I'm developing a feature to differentiate between weekday and weekend order statuses.

In [8]:
orders["is_weekend"] = orders["purchase_weekday"].isin(
    ["Saturday", "Sunday"]
)

In [9]:
orders["is_weekend"].value_counts()

is_weekend
False    76594
True     22847
Name: count, dtype: int64

In [10]:
orders["is_weekend"].value_counts(normalize=True)

is_weekend
False    0.770246
True     0.229754
Name: proportion, dtype: float64

I'm investigating the question, "How many hours did it take for the order to be confirmed?" The relationship between order confirmation efficiency, delivery performance, and customer satisfaction can be examined.

The distribution appears to be skewed to the right. This means that the vast majority of orders were confirmed quickly, but orders confirmed very late have pushed the average upwards. The order confirmed in 4509 hours may have been recorded late in the system, there may have been a data entry problem, a data error, or the payment may have been confirmed months later.

In [11]:
orders["approval_time_hours"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

orders["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

I left 160 NaN in the order_approved_at column during the cleaning phase. Therefore, it's normal to see this value here as well.;

In [12]:
orders["approval_time_hours"].isna().sum()

np.int64(160)

In [13]:
orders.loc[
    orders["approval_time_hours"] < 0,
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours"
    ]
]

,order_purchase_timestamp,order_approved_at,approval_time_hours


I'm creating a shipping_time feature to analyze the distribution of orders for shipment/order preparation.

Some orders appear to have been shipped before confirmation. This is contrary to normal workflow. 125 days for shipping is also a long time. There may have been a stock issue, the system may have updated late, or the order may have been delayed.

In [14]:
orders["shipping_time_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_approved_at"]
).dt.total_seconds() / (60 * 60 * 24)

orders["shipping_time_days"].describe()

count    97644.000000
mean         2.805038
std          3.549427
min       -171.219005
25%          0.875509
50%          1.818397
75%          3.580469
max        125.762569
Name: shipping_time_days, dtype: float64

I am reviewing orders with incorrect delivery times. Some confirmation times (such as 23:31) are duplicated. In this case, the confirmation time may not be the actual confirmation time; it may have been assigned or rounded off by the system later.

In [15]:
orders.loc[
    orders["shipping_time_days"] < 0,
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "shipping_time_days"
    ]
]

,order_approved_at,order_delivered_carrier_date,shipping_time_days
15,2018-06-12 23:31:02,2018-06-11 14:54:00,-1.359051
64,2018-04-24 18:25:22,2018-04-23 19:19:14,-0.962593
199,2018-07-26 23:31:53,2018-07-24 12:57:00,-2.440891
210,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.005475
415,2018-07-27 23:31:09,2018-07-24 14:03:00,-3.394549
...,...,...,...
99091,2018-07-05 16:17:59,2018-07-05 14:11:00,-0.088183
99230,2018-07-05 16:32:52,2018-07-03 12:57:00,-2.149907
99266,2018-02-04 23:31:46,2018-01-31 18:11:58,-4.222083
99377,2018-04-24 19:26:10,2018-04-23 17:18:40,-1.088542


A subset of orders shows negative shipping times because the recorded carrier pickup timestamp precedes the recorded approval timestamp. Since the approval times of these orders are also unusually long, this likely indicates timestamp inconsistencies or data quality issues rather than actual business events. The records are retained for transparency and will be considered during later analyses. This is derived data quality issue.

In [16]:
orders.loc[
    orders["shipping_time_days"] < 0,
    "approval_time_hours"
].describe()

count    1359.000000
mean       54.519698
std        49.042012
min         0.124444
25%        20.629722
50%        46.232778
75%        78.489306
max       291.410556
Name: approval_time_hours, dtype: float64

In [17]:
orders.loc[
    orders["shipping_time_days"] < 0,
    "order_status"
].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

1797 NaNs are normal. There were 160 NaN in the order_approved_at column and 1783 NaN in the order_delivered_carrier_date column. Therefore, it is normal to have a NaN value that is more than 1783, but less than the sum of these two numbers.

In [18]:
orders["shipping_time_days"].isna().sum()


np.int64(1797)

I'm examining the delivery time values, which are one of the most important features in the orders table. This feature measures the customer's end-to-end delivery experience and supports analytics related to delivery performance, customer satisfaction, and operational efficiency.

In [19]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

orders["delivery_time_days"].describe()

count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_time_days, dtype: float64

These NaN values ​​are normal because there were orders where we left NaN in the order_delivered_customer_date column during the data cleanup process.

In [20]:
orders["delivery_time_days"].isna().sum()

np.int64(2965)

In [21]:
orders.loc[
    orders["delivery_time_days"] < 0,
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_time_days"
    ]
]

,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days


So far, I've examined time-related properties individually in separate code snippets for illustrative purposes. However, these calculations can be done much faster using functions.

I look at the difference between the estimated delivery date the business tells the customer and the actual delivery date. This allows me to mark some orders as "late delivery".

Overall, the delivery was made earlier than expected. The -146 and +188 values ​​here are likely outliers. They will be investigated further.

In [22]:
orders["estimated_delivery_gap_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders["estimated_delivery_gap_days"].describe()

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: estimated_delivery_gap_days, dtype: float64

In [23]:
orders["estimated_delivery_gap_days"].isna().sum()

np.int64(2965)

In [24]:
orders["estimated_delivery_gap_days"].value_counts().head(10)

estimated_delivery_gap_days
-12.385602    5
-15.084213    4
-7.181331     4
-13.144051    4
-13.283183    4
-8.185579     4
-7.216123     4
-13.286227    4
-9.091238     4
-9.213519     4
Name: count, dtype: int64

Based on this difference, we will derive an "is_late_delivery" feature for late delivery.

In [25]:
orders["is_late_delivery"] = (
    orders["estimated_delivery_gap_days"] > 0
)
orders["is_late_delivery"]

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Name: is_late_delivery, Length: 99441, dtype: bool

I'm looking at the number of orders that were delivered late.

In [26]:
orders["is_late_delivery"].value_counts()

is_late_delivery
False    91614
True      7827
Name: count, dtype: int64

A new categorical feature, `purchase_period`, was created by grouping purchase hours into four time periods: Night, Morning, Afternoon, and Evening.

This feature provides a more interpretable representation of customer purchasing behavior and supports analyses of shopping patterns across different periods of the day.

In [27]:
def get_purchase_period(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

orders["purchase_period"] = orders["purchase_hour"].apply(get_purchase_period)
orders["purchase_period"].value_counts()

purchase_period
Afternoon    38361
Evening      34100
Morning      22240
Night         4740
Name: count, dtype: int64

ORDERS FEATURE ENGINEERING BİTTİ

PRODUCTS FEATURE ENGINEERING


I completed the orders table. Now I will perform the necessary feature engineering operations in the products table.

From a logistical perspective, not only the weight but also the volume occupied by a product is important. Therefore, I take that into account.

Since there are different categories, the product volume range can be wide. There's nothing unusual about it.

In [28]:
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

products["product_volume_cm3"].describe()

count     32949.000000
mean      16564.096695
std       27057.041650
min         168.000000
25%        2880.000000
50%        6840.000000
75%       18480.000000
max      296208.000000
Name: product_volume_cm3, dtype: float64

The product dimensions already had two NaN values. Therefore, I was expecting a similar result here.

In [29]:
products["product_volume_cm3"].isna().sum()

np.int64(2)

In [30]:
products.loc[
    products["product_volume_cm3"] <= 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3


A product can have a large volume but a small weight, or vice versa. Therefore, density calculations are necessary. This can be useful in logistics, packaging, storage, and transportation analyses.

In general, product densities rarely exceeded 0.2 g/cm³. However, the maximum value was 85.2 g/cm³. This could be a product with very high weight or one that takes up little space. Or it could be a data entry error (for example, mistakenly entering 1 cm for the length). I will investigate this situation later in EDA.

In [31]:
products["product_density_g_cm3"] = (
    products["product_weight_g"] /
    products["product_volume_cm3"]
)

products["product_density_g_cm3"].describe()

count    32945.000000
mean         0.203714
std          1.009329
min          0.000220
25%          0.066204
50%          0.116550
75%          0.195869
max         85.227273
Name: product_density_g_cm3, dtype: float64

In [32]:
products["product_density_g_cm3"].isna().sum()

np.int64(6)

In [33]:
products.loc[
    products["product_density_g_cm3"] <= 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,product_density_g_cm3


This feature indicates whether a product has all essential listing information available.

A product is considered complete if the following fields are present:

- Product category
- Product name length
- Product description length
- Product photo count

This allows me to analyze whether there is a relationship between having complete product information and sales.

In [34]:
products["has_complete_listing"] = (
    products[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
        ]
    ]
    .notna()
    .all(axis=1)
)
products["has_complete_listing"].value_counts()

has_complete_listing
True     32341
False      610
Name: count, dtype: int64

order_items İÇİN FEATURE BAŞLANGIÇ

Normally, the order_items_clean table has 112,651 rows, so it's normal to see fewer unique entries here. This is because order_item_id is not a primary key on its own; both order_id and order_item_id together form the primary key.

In [35]:
order_items.nunique()

order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

I'm summing the price and shipping costs to calculate the total amount. This will be useful in many analyses. The fact that the average is significantly larger than the median suggests the distribution may be right-skewed. A small number of expensive orders have pushed the average upward.

In [36]:
order_items["item_total_cost"] = (
    order_items["price"] +
    order_items["freight_value"]
)

order_items["item_total_cost"].describe()

count    112650.000000
mean        140.644059
std         190.724394
min           6.080000
25%          55.220000
50%          92.320000
75%         157.937500
max        6929.310000
Name: item_total_cost, dtype: float64

In [37]:
order_items["item_total_cost"].isna().sum()

np.int64(0)

The percentage of an order that includes shipping costs is important. Therefore, I'm creating a feature that calculates the shipping rate. This could influence customer behavior. I will evaluate this in the analytics section.

A maximum value of 0.98 is a high rate for the shipping rate. The product price might be very low in these orders. This will be evaluated in detail.

In [38]:
order_items["freight_ratio"] = (
    order_items["freight_value"] /
    order_items["item_total_cost"]
)

order_items["freight_ratio"].describe()

count    112650.000000
mean          0.213364
std           0.129498
min           0.000000
25%           0.118192
50%           0.187887
75%           0.282144
max           0.963283
Name: freight_ratio, dtype: float64

In [39]:
(order_items["freight_ratio"] < 0).sum()

np.int64(0)

I'm reviewing the top 10 orders with the highest shipping rates. The product prices appear very low in these orders. The price might genuinely be low, for example, a discount might have been applied, but there could also be a problem with the data entry. This will be investigated.

In [40]:
order_items.nlargest(
    10,
    "freight_ratio"
)[
    [
        "price",
        "freight_value",
        "item_total_cost",
        "freight_ratio"
    ]
]

,price,freight_value,item_total_cost,freight_ratio
87081,0.85,22.30,23.15,0.963283
27652,0.85,18.23,19.08,0.955451
48625,0.85,18.23,19.08,0.955451
110535,9.90,121.22,131.12,0.924497
94495,4.99,37.04,42.03,0.881275
57297,1.20,7.89,9.09,0.867987
57298,1.20,7.89,9.09,0.867987
57299,1.20,7.89,9.09,0.867987
57300,1.20,7.89,9.09,0.867987
57301,1.20,7.89,9.09,0.867987


I'm creating a feature to see how many items are in an order. This feature will allow me to analyze customers who place large orders, those who buy single items, and the relationships between order size and sales/delivery times, etc.

75% of orders contain 1 item, but some contain a high number of items, such as 21.

In [41]:
order_items["items_in_order"] = (
    order_items
    .groupby("order_id")["order_item_id"]
    .transform("count")
)

order_items["items_in_order"].describe()

count    112650.000000
mean          1.395668
std           1.120101
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
Name: items_in_order, dtype: float64

In [42]:
order_items["items_in_order"].value_counts().sort_index()

items_in_order
1     88863
2     15032
3      3966
4      2020
5      1020
6      1188
7       154
8        64
9        27
10       80
11       44
12       60
13       13
14       28
15       30
20       40
21       21
Name: count, dtype: int64

As you can see, there can be multiple products in a single order. In such cases, I add a new column showing the total amount of that order. At this stage, I'm excluding the shipping cost.

There is a right-skewed distribution because the mean is larger than the median, the standard deviation is even larger than the mean, and the maximum value is approximately 88 times the mean. A small number of high-priced orders have pushed the mean upward.

In [43]:
order_items["order_total_price"] = (
    order_items
    .groupby("order_id")
    ["price"]
    .transform("sum")
)

order_items["order_total_price"].describe()

count    112650.000000
mean        152.959595
std         257.339594
min           0.850000
25%          49.000000
50%          91.550000
75%         164.990000
max       13440.000000
Name: order_total_price, dtype: float64

Next, I create a feature to examine the answer to the question, "How much was paid in total for shipping in an order?" The maximum value of 1794 is very high compared to the general situation. The destination might be very far away, the product might be heavy or bulky, or it might be a premium/express shipment.

In [44]:
order_items["order_total_freight"] = (
    order_items
    .groupby("order_id")["freight_value"]
    .transform("sum")
)
order_items["order_total_freight"].describe()

count    112650.000000
mean         27.285033
std          33.218827
min           0.000000
25%          14.292500
50%          18.160000
75%          29.220000
max        1794.960000
Name: order_total_freight, dtype: float64

I had already calculated the total product price and total shipping cost per order. Now I'm combining these to calculate the total amount paid per order.

In [45]:
order_items["order_total_cost"] = (
    order_items["order_total_price"] +
    order_items["order_total_freight"]
)

order_items["order_total_cost"].describe()

count    112650.000000
mean        180.244628
std         272.829911
min           9.590000
25%          65.620000
50%         114.440000
75%         195.337500
max       13664.080000
Name: order_total_cost, dtype: float64

There may be one or more vendors in an order. I am producing a feature for this because in the future, analysis can be made between the number of sellers and variables such as shipping time, satisfaction status, cancellation rate.

In [46]:
order_items["sellers_in_order"] = (
    order_items
    .groupby("order_id")["seller_id"]
    .transform("nunique")
)

order_items["sellers_in_order"].describe()

count    112650.000000
mean          1.029898
std           0.186050
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           5.000000
Name: sellers_in_order, dtype: float64

In [47]:
order_items["sellers_in_order"].value_counts().sort_index()

sellers_in_order
1    109547
2      2876
3       202
4        12
5        13
Name: count, dtype: int64

I'm creating a new feature to analyze how free shipping affects sales, customer satisfaction, ratings, etc.

In [48]:
order_items["is_free_shipping"] = (
    order_items["freight_value"] == 0
)

order_items["is_free_shipping"].value_counts()

is_free_shipping
False    112267
True        383
Name: count, dtype: int64

I'm verifying the situation with this validation.

In [49]:
order_items.loc[
    order_items["is_free_shipping"],
    ["price", "freight_value", "order_total_price"]
].head(10)

,price,freight_value,order_total_price
114,99.9,0.0,99.9
258,69.9,0.0,69.9
483,99.9,0.0,99.9
508,53.9,0.0,107.8
509,53.9,0.0,107.8
1784,69.9,0.0,69.9
2232,99.9,0.0,99.9
2714,110.0,0.0,110.0
3224,106.9,0.0,106.9
3378,69.9,0.0,69.9


I'm creating an average product price. Based on this information, I can observe how delivery time, return/review behavior, and rating levels are affected.

In [50]:
order_items["average_item_price"] = (
    order_items["order_total_price"] /
    order_items["items_in_order"]
)

order_items["average_item_price"].describe()

count    112650.000000
mean        120.653739
std         183.075301
min           0.850000
25%          39.950000
50%          74.990000
75%         134.900000
max        6735.000000
Name: average_item_price, dtype: float64

In [51]:
order_items.nlargest(
    10,
    "average_item_price"
)[
    [
        "items_in_order",
        "order_total_price",
        "average_item_price"
    ]
]

,items_in_order,order_total_price,average_item_price
3556,1,6735.00,6735.00
112233,1,6729.00,6729.00
107841,1,6499.00,6499.00
74336,1,4799.00,4799.00
11249,1,4690.00,4690.00
62086,1,4590.00,4590.00
29193,1,4399.87,4399.87
45843,1,4099.99,4099.99
78310,1,4059.00,4059.00
59137,1,3999.90,3999.90


customers için FEATURE ENGINEERING

The number of customer_unique_ids is low, meaning the same person (customer_id) placed orders with multiple customer_ids. This information will be useful in future analyses such as Customer Lifetime Value.

In [52]:
customers.nunique()

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

First, I look for customer_unique_ids that have multiple customer_ids. This allows me to track repeat customers.

In [53]:
customer_counts = (
    customers
    .groupby("customer_unique_id")["customer_id"]
    .transform("count")
)

customers["is_repeat_customer"] = customer_counts > 1

customer_counts.describe()

count    99441.000000
mean         1.079223
std          0.396154
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: customer_id, dtype: float64

In [54]:
customers.groupby("customer_unique_id").size().value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [55]:
customers["is_repeat_customer"].value_counts()

is_repeat_customer
False    93099
True      6342
Name: count, dtype: int64

First, I convert the states, based on Brazil's map and geographical information, into major regions of Brazil. This allows for analysis by region instead of by 27 different states.

In [56]:
region_map = {
    "AC": "North",
    "AP": "North",
    "AM": "North",
    "PA": "North",
    "RO": "North",
    "RR": "North",
    "TO": "North",

    "AL": "Northeast",
    "BA": "Northeast",
    "CE": "Northeast",
    "MA": "Northeast",
    "PB": "Northeast",
    "PE": "Northeast",
    "PI": "Northeast",
    "RN": "Northeast",
    "SE": "Northeast",

    "DF": "Central-West",
    "GO": "Central-West",
    "MT": "Central-West",
    "MS": "Central-West",

    "ES": "Southeast",
    "MG": "Southeast",
    "RJ": "Southeast",
    "SP": "Southeast",

    "PR": "South",
    "RS": "South",
    "SC": "South"
}

I'm creating a feature called `customer_region` for each customer. This will allow for region-based analysis and comparisons in the future.

In [57]:
customers["customer_region"] = (
    customers["customer_state"]
    .map(region_map)
)

Since Brazil's most populous and economically significant states, such as São Paulo (SP), Rio de Janeiro (RJ), and Minas Gerais (MG), are located in the Southeast region, it is normal for customer density to be concentrated there.

In [58]:
customers["customer_region"].value_counts()

customer_region
Southeast       68266
South           14148
Northeast        9394
Central-West     5782
North            1851
Name: count, dtype: int64

I'm running a null check to make sure I haven't forgotten any states. All customers are grouped together.

In [59]:
customers["customer_region"].isna().sum()

np.int64(0)

SELLERS FEATURE ENGINEERING BAŞLANGIÇ

In [60]:
sellers.shape

(3095, 4)

In [61]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [62]:
sellers.nunique()

seller_id                 3095
seller_zip_code_prefix    2246
seller_city                611
seller_state                23
dtype: int64

Just like with customers, I can also segment sellers by region. This allows me to analyze whether sellers and customers are in the same region and how this affects delivery time or shipping costs.

In [63]:
sellers["seller_region"] = sellers["seller_state"].map(region_map)

In [64]:
sellers["seller_region"].value_counts()

seller_region
Southeast       2287
South            668
Central-West      79
Northeast         56
North              5
Name: count, dtype: int64

I confirm that I was able to include all vendors in the grouping.

In [65]:
sellers["seller_region"].isna().sum()

np.int64(0)

`seller_order_count` represents the number of unique orders associated with each seller.

This feature provides a simple measure of seller activity on the platform and can later be used to compare high-volume and low-volume sellers in terms of delivery performance, freight costs, customer satisfaction, and other business metrics.

In [66]:
seller_order_counts = (
    order_items
    .groupby("seller_id")["order_id"]
    .nunique()
)

sellers["seller_order_count"] = (
    sellers["seller_id"].map(seller_order_counts)
)


The distribution is strongly right-skewed. Most sellers are associated with relatively few orders, while a small number of sellers account for a very large number of orders. This suggests substantial variation in seller activity and may be useful for identifying high-volume sellers in later analysis.

In [67]:
sellers["seller_order_count"].describe()

count    3095.000000
mean       32.313409
std       105.139763
min         1.000000
25%         2.000000
50%         6.000000
75%        21.500000
max      1854.000000
Name: seller_order_count, dtype: float64

In [68]:
sellers["seller_order_count"].isna().sum()

np.int64(0)

In [69]:
sellers["seller_order_count"].sort_values(ascending=False).head(10)

797     1854
2463    1806
1413    1706
474     1404
1873    1314
390     1287
2207    1160
987     1146
2617    1132
2345    1080
Name: seller_order_count, dtype: int64

ORDER PAYMENTS FEATURE BAŞLANGIÇ

In [70]:
order_payments.shape

(103886, 5)

In [71]:
order_payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    2
payment_value           0
dtype: int64

In [72]:
order_payments.nunique()

order_id                99440
payment_sequential         29
payment_type                5
payment_installments       23
payment_value           29077
dtype: int64

`order_payments` tablosu, bazı siparişler için birden fazla ödeme kaydı içermektedir. Ana analiz sipariş düzeyinde yapılacağından, ödeme düzeyindeki kayıtlar sipariş düzeyinde bir özet tabloya toplanmıştır.

- `total_payment_value`: Siparişle ilişkili toplam ödeme tutarı.

- `num_payment_methods`: Sipariş için kullanılan benzersiz ödeme yöntemlerinin sayısı.

- `max_installments`: Siparişle ilişkili maksimum taksit sayısı.

Orijinal `order_payments` tablosu korunmuştur. `payment_summary`, sipariş başına bir satır içeren ayrı bir türetilmiş tablodur.

In [73]:
payment_summary = order_payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    num_payment_methods=("payment_type", "nunique"),
    max_installments=("payment_installments", "max")
).reset_index()

Since the order_id table has 99440 records, it's normal for this newly created table to have the same number of rows.

In [74]:
payment_summary.shape

(99440, 4)

Now we have such a clear picture of the payment status for all orders.

In [75]:
payment_summary.head()

,order_id,total_payment_value,num_payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3.0


Keeping a binary feature of whether payments are made in installments or not can be useful for businesses. This allows me to analyze characteristics like average basket value and customer delivery behavior for both installment and single-payment purchases.

In [76]:
payment_summary["is_installment_payment"] = (
    payment_summary["max_installments"] > 1
)

payment_summary["is_installment_payment"].value_counts()

is_installment_payment
True     51170
False    48270
Name: count, dtype: int64

`max_installments` contains 2 missing values out of 99,440 orders. Since the proportion of missing values is extremely low and there is no reliable basis for imputing the installment count, these values were left as `NaN`.

The missing values are retained rather than replaced with an assumed installment count to avoid introducing unsupported information.

When using `is_installment_payment`, these two orders should be treated cautiously because their installment status is unknown rather than definitively non-installment.

In [77]:
payment_summary["max_installments"].isna().sum()

np.int64(2)

ORDER REVIEWS FEATURE ENGINEERING BAŞLANGIÇ

In [78]:
order_reviews.shape

(98673, 7)

In [79]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87121
review_comment_message     57897
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [80]:
order_reviews.nunique()

review_id                  98114
order_id                   98673
review_score                   5
review_comment_title        4525
review_comment_message     36064
review_creation_date         636
review_answer_timestamp    97953
dtype: int64

I'm creating a feature related to response time, covering the time between a customer creating a review and the seller/business responding to it. This will allow me to perform various analyses, such as whether businesses that respond quickly to reviews have different ratings, whether low-rated reviews are responded to more quickly, or whether there are businesses with very delayed responses. 

To do this, I'm first converting the data type.

In [81]:
order_reviews["review_creation_date"] = pd.to_datetime(
    order_reviews["review_creation_date"]
)

order_reviews["review_answer_timestamp"] = pd.to_datetime(
    order_reviews["review_answer_timestamp"]
)

order_reviews["response_time_hours"] = (
    order_reviews["review_answer_timestamp"]
    - order_reviews["review_creation_date"]
).dt.total_seconds() / 3600

In [82]:
order_reviews["response_time_hours"].describe()

count    98673.000000
mean        75.590394
std        237.828473
min          2.141389
25%         24.107222
50%         40.198333
75%         74.449444
max      12448.781111
Name: response_time_hours, dtype: float64

In addition to rating customers, I also check whether they have left a written review, and I've created a column for this purpose.

In [83]:
order_reviews["has_comment"] = (
    order_reviews["review_comment_message"].notna()
).astype(int)

order_reviews["has_comment"].value_counts(normalize=True)

has_comment
0    0.586756
1    0.413244
Name: proportion, dtype: float64

I'm calculating the length of the comments written. Longer comments generally tend to convey a stronger emotion (positive or negative). I'm specifically using .fillna("") here because I want 0 characters instead of NaN for non-comments.

In [84]:
order_reviews["review_length"] = (
    order_reviews["review_comment_message"]
    .fillna("")
    .str.len()
)

order_reviews["review_length"].describe()

count    98673.000000
mean        28.358822
std         48.362591
min          0.000000
25%          0.000000
50%          0.000000
75%         42.000000
max        208.000000
Name: review_length, dtype: float64

CLOSED DEALS FE BAŞLANGIÇ

In [85]:
closed_deals.isna().sum()

mql_id                             0
seller_id                          0
sdr_id                             0
sr_id                              0
won_date                           0
business_segment                   1
lead_type                          6
lead_behaviour_profile           177
has_company                      779
has_gtin                         778
average_stock                    776
business_type                     10
declared_product_catalog_size    773
declared_monthly_revenue           0
dtype: int64

In [86]:
closed_deals.nunique()

mql_id                           842
seller_id                        842
sdr_id                            32
sr_id                             22
won_date                         824
business_segment                  33
lead_type                          8
lead_behaviour_profile             9
has_company                        2
has_gtin                           2
average_stock                      6
business_type                      3
declared_product_catalog_size     33
declared_monthly_revenue          27
dtype: int64

In [87]:
closed_deals.columns

Index(['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date',
       'business_segment', 'lead_type', 'lead_behaviour_profile',
       'has_company', 'has_gtin', 'average_stock', 'business_type',
       'declared_product_catalog_size', 'declared_monthly_revenue'],
      dtype='str')

The year in which the deal was won. ay ve çeyrek dönem için 3 yeni feature oluşturdum.

These features were created to enable future analysis of seasonality and sales performance over time, such as identifying periods with higher deal conversion activity.

The distribution shows that the dataset is heavily concentrated in 2018, with only 3 deals recorded in 2017. Among the quarters, Q1 and Q2 account for the majority of closed deals, while deal counts decrease considerably in Q3 and Q4. However, this pattern should be interpreted cautiously because the dataset does not cover a complete and balanced period across all months.

In [88]:
closed_deals["won_date"] = pd.to_datetime(
    closed_deals["won_date"]
)

closed_deals["won_year"] = closed_deals["won_date"].dt.year
closed_deals["won_month"] = closed_deals["won_date"].dt.month
closed_deals["won_quarter"] = closed_deals["won_date"].dt.quarter

In [89]:
closed_deals["won_year"].value_counts().sort_index()


won_year
2017      3
2018    839
Name: count, dtype: int64

In [90]:
closed_deals["won_month"].value_counts().sort_index()

won_month
1      73
2     113
3     147
4     207
5     122
6      57
7      37
8      33
9      23
10     21
11      6
12      3
Name: count, dtype: int64

In [91]:
closed_deals["won_quarter"].value_counts().sort_index()

won_quarter
1    333
2    386
3     93
4     30
Name: count, dtype: int64

I'm creating a feature to answer the question, "Is there declared income information?". The "declared_monthly_revenue" column in the table doesn't already have a null value, so there's no difficulty in classification. This feature will allow me to perform analyses such as "Are vendors declaring high income coming from a different marketing source?"

Most closed deals (797) do not have a positive declared revenue, while only 45 sellers reported revenue above zero. The feature is therefore highly imbalanced, but it may still be useful for distinguishing the small group of sellers with declared revenue from the majority without it.

In [92]:
closed_deals["has_declared_revenue"] = (
    closed_deals["declared_monthly_revenue"] > 0
)

closed_deals["has_declared_revenue"].value_counts()

has_declared_revenue
False    797
True      45
Name: count, dtype: int64

MARKETING LEADS FE BAŞLANGIÇ

In [93]:
marketing_leads.shape

(8000, 4)

In [94]:
marketing_leads.isna().sum()

mql_id                 0
first_contact_date     0
landing_page_id        0
origin                60
dtype: int64

In [95]:
marketing_leads.nunique()

mql_id                8000
first_contact_date     336
landing_page_id        495
origin                  10
dtype: int64

In [96]:
marketing_leads.columns

Index(['mql_id', 'first_contact_date', 'landing_page_id', 'origin'], dtype='str')

The `first_contact_date` column represents the date when the lead was first contacted, rather than the date when the deal was successfully closed.
These features can be used to analyze the timing and potential seasonality of marketing lead generation.

In [97]:
marketing_leads["first_contact_date"] = pd.to_datetime(
    marketing_leads["first_contact_date"]
)

marketing_leads["contact_year"] = (
    marketing_leads["first_contact_date"].dt.year
)

marketing_leads["contact_month"] = (
    marketing_leads["first_contact_date"].dt.month
)

marketing_leads["contact_quarter"] = (
    marketing_leads["first_contact_date"].dt.quarter
)

In [98]:
marketing_leads["contact_year"].value_counts().sort_index()

contact_year
2017    2002
2018    5998
Name: count, dtype: int64

This could be a genuine seasonal pattern, but let's not immediately interpret it that way. The fact that there were only 4 records in June could also be related to data collection/ETL or historical context.

In [99]:
marketing_leads["contact_month"].value_counts().sort_index()

contact_month
1     1141
2     1028
3     1174
4     1352
5     1303
6        4
7      239
8      386
9      312
10     416
11     445
12     200
Name: count, dtype: int64

In [100]:
marketing_leads["contact_quarter"].value_counts().sort_index()

contact_quarter
1    3343
2    2659
3     937
4    1061
Name: count, dtype: int64

In [101]:
marketing_leads["origin"].value_counts(dropna=False)

origin
organic_search       2296
paid_search          1586
social               1350
unknown              1099
direct_traffic        499
email                 493
referral              284
other                 150
display               118
other_publicities      65
NaN                    60
Name: count, dtype: int64

The `is_organic` feature indicates whether a lead originated from `organic_search`.

This binary feature allows us to compare organically acquired leads with leads from other sources in later funnel analyses, particularly when evaluating differences in conversion and sales performance.

Note that `False` represents all non-organic categories, including paid search, social, referral, unknown, and other sources; it should not be interpreted as "paid" traffic specifically.

In [102]:
marketing_leads["is_organic"] = (
    marketing_leads["origin"] == "organic_search"
)

marketing_leads["is_organic"].value_counts()

is_organic
False    5704
True     2296
Name: count, dtype: int64

MERGE SİZ SON

In [103]:
import os

os.makedirs("../data/feature_engineered", exist_ok=True)

orders.to_csv("../data/feature_engineered/orders_fe.csv", index=False)
order_items.to_csv("../data/feature_engineered/order_items_fe.csv", index=False)
customers.to_csv("../data/feature_engineered/customers_fe.csv", index=False)
sellers.to_csv("../data/feature_engineered/sellers_fe.csv", index=False)
products.to_csv("../data/feature_engineered/products_fe.csv", index=False)
order_payments.to_csv("../data/feature_engineered/order_payments_fe.csv", index=False)
order_reviews.to_csv("../data/feature_engineered/order_reviews_fe.csv", index=False)
closed_deals.to_csv("../data/feature_engineered/closed_deals_fe.csv", index=False)
marketing_leads.to_csv("../data/feature_engineered/marketing_leads_fe.csv", index=False)